In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("NASCAR") \
    .master("local[4]") \
    .getOrCreate()

# Load once
df = spark.read.csv("cup_series_transformed.csv", header=True, inferSchema=True)
df.cache()
df.filter((F.col("Season") == 2025) & (F.col("Race") == 7)).show(5)

In [ ]:
# driver dimens
from pyspark.sql.functions import round
driver_df = df.groupBy("Season", "Driver")\
            .agg(
                F.count("Race").alias("Races"),
                F.sum("Top_5").alias("Top5"),
                F.sum("Top_10").alias("Top10"), 
                F.sum("Win").alias("Wins"),
                F.round(F.avg("Finish"), 2).alias("Avg_Finish"),
                F.sum("Pts").alias("Total_Points"),
                F.sum("Laps").alias("Total_Laps"),
                F.sum("Led").alias("Total_Laps_Led"),
                F.sum("Lead_Lap").alias("Lead_Lap_Finishes"),
                F.round(F.avg("Start"), 2).alias("Avg_Start"),
                F.min("Finish").alias("Best_Finish"),
                F.max("Finish").alias("Worst_Finish")        
            )\
            .withColumn("Top_5 Pct", F.round(F.expr("try_divide(Top5, Races)") * 100, 2))\
            .withColumn("Top_10 Pct", F.round(F.expr("try_divide(Top10, Races)") * 100, 2)) \
            .withColumn("Win_Pct", F.round(F.expr("try_divide(Wins, Races)") * 100, 2)) \
            .withColumn("Laps_Led_Pct", F.round(F.expr("try_divide(Total_Laps_Led, Total_Laps)") * 100, 2))
            

driver_df.filter(F.col("Season") == 2023).show(5)
driver_df.cache()           

# F.round(F.expr("try_divide(Total_Laps_Led, Total_Laps)") * 100, 2)







In [ ]:
driver_df = df.

In [ ]:
import pandas as pd

# Convert to Pandas and save (works perfectly on Windows)
pandas_df = driver_df.toPandas()
pandas_df.to_csv("driver_table/driver_season_stats.csv", index=False)
print(f"✓ Saved {len(pandas_df):,} rows to driver_season_stats.csv")

In [ ]:
#stage racing
#driver_stage_df = df.groupBy("Season", "Driver")\
#                    .withColumn()

#driver_df = driver_df.drop("Total_Stage_Points")
# View the result
